# Per-sample segmentation stats

Reads the deconvolved (preprocessed) image of each sample, applies the **same** normalization and
3D adapted-Sauvola segmentation as the pipeline, and prints for every sample:

```
### <sample>
# v_low  = ...        (robust background floor)
# v_high = ...        (high percentile of the above-background signal)
# max    = ...        (raw image maximum)
# nuclei = ...        (connected components in the labelled nuclei mask)
# shape  = ...        (deconvolved image shape)
# connexin regions = ...    (3D connected components after Sauvola)
# single voxel plaques = ... (size-1 regions = threshold noise)
```

All settings (Sauvola window/k, normalization percentile) come from `run_pipeline.COMMON`, and the
nuclei mask + crop come from `run_pipeline.SAMPLES`, so the numbers match a full `run_pipeline.py` run.
Segmentation runs on the **full** deconvolved frame here (no segmentation crop), so the shape is the
raw deconvolved shape.

In [ ]:
from pathlib import Path
import sys
import gc

import numpy as np
import pandas as pd
from skimage import io

BASE_DIR = Path.cwd().parent
sys.path.append(str(BASE_DIR))              # run_pipeline (repo root)
sys.path.append(str(BASE_DIR / 'src'))      # src modules

from run_pipeline import COMMON, resolve_config, _apply_crop
from threshold import estimate_normalization_bounds, savola_3D_image
from localization import label_connexin_regions_3d
from nuclei_assignment import get_nuclei_centerpoints

RESULTS_DIR = BASE_DIR / 'results' / 'sample_segmentation_stats'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# (sample name in SAMPLES, deconvolved image path) -- edit paths here if they differ
SAMPLE_INPUTS = [
    ('sample_1', BASE_DIR / 'data/preprocessed/background_removed_and_deconvolved_img.tiff'),
    ('sample_2', BASE_DIR / 'data/raw/sample_2/preprocessed/background_removed_and_deconvolved_img.tiff'),
    ('sample_3', BASE_DIR / 'data/raw/sample_3/preprocessed/background_removed_and_deconvolved_img.tiff'),
]

In [ ]:
def analyze_sample(name, deconv_path):
    """Segment one sample's deconvolved image and return its stats dict."""
    cfg = resolve_config(name)

    image = io.imread(deconv_path)
    shape = tuple(image.shape)
    img_max = float(image.max())

    # normalization bounds (same estimator as the pipeline)
    v_low, v_high = estimate_normalization_bounds(
        image, low_sigma=COMMON['norm_low_sigma'], high_percentile=COMMON['norm_high_pct'])

    # 3D adapted Sauvola (blockwise; normalizes internally with these bounds)
    binary = savola_3D_image(
        image, v_low=v_low, v_high=v_high,
        window_size=COMMON['sauvola_window'], k=COMMON['sauvola_k'], r=COMMON['sauvola_r'],
        threeD=True)
    del image; gc.collect()

    # label once, derive both region count and single-voxel count from it
    labeled, n_regions = label_connexin_regions_3d(binary)
    del binary; gc.collect()
    sizes = np.bincount(labeled.ravel())[1:]      # drop background label 0
    n_single = int((sizes == 1).sum())
    del labeled; gc.collect()

    # nuclei: connected components in the labelled nuclei mask (with the configured crop)
    n_nuclei = np.nan
    nuclei_path = BASE_DIR / cfg['nuclei_mask']
    if nuclei_path.exists():
        mask = _apply_crop(io.imread(nuclei_path), *cfg['nuclei_crop'])
        n_nuclei = int(len(get_nuclei_centerpoints(mask)))
        del mask; gc.collect()
    else:
        print(f"  (nuclei mask not found: {nuclei_path})")

    return {'sample': name, 'v_low': round(v_low, 1), 'v_high': round(v_high, 1),
            'max': round(img_max, 2), 'nuclei': n_nuclei, 'shape': shape,
            'connexin_regions': int(n_regions), 'single_voxel_plaques': n_single}


def print_block(r):
    print(f"### {r['sample']}")
    print(f"# v_low = {r['v_low']}")
    print(f"# v_high = {r['v_high']}")
    print(f"# max = {r['max']}")
    print(f"# nuclei = {r['nuclei']}")
    print(f"# shape = {r['shape']}")
    print(f"# connexin regions = {r['connexin_regions']}")
    print(f"# single voxel plaques = {r['single_voxel_plaques']}")
    print()

## Run all samples

In [ ]:
results = []
for name, path in SAMPLE_INPUTS:
    if not Path(path).exists():
        print(f"skipping {name}: no deconvolved image at {path}\n")
        continue
    print(f"processing {name} ...")
    r = analyze_sample(name, path)
    print_block(r)
    results.append(r)
    gc.collect()

assert results, "No images found - check the paths in SAMPLE_INPUTS."


## Summary table

In [ ]:
summary = pd.DataFrame(results).set_index('sample')
summary.to_csv(RESULTS_DIR / 'segmentation_stats.csv')
print(f"saved {RESULTS_DIR / 'segmentation_stats.csv'}")
summary